# Uniform mode sandbox

This notebook is intentionally **not** wired to production implementations except for preprocessing/layout utilities and optional Triton kernel wrappers. It is meant for quick experiments before editing the real codebase.

The notebook-local functions below copy the uniform-mode logic for:

| Computation | Notebook behavior |
| --- | --- |
| Parameter extraction | Editable PyTorch copy of `extract_parameters_uniform` |
| `E_fixed_point` / `E_step` | Editable PyTorch copy using sparse `ancestors_T` |
| Uniform `Pibar` in Pi forward | Editable PyTorch copy using `exp2`, row sum, and `Pi_exp @ ancestors_T` |
| Cross-clade DTS forward | Editable PyTorch copy, with optional production Triton wrapper |
| Wave self-loop forward | Editable PyTorch copy, with optional production Triton wrapper |
| Likelihood | Editable PyTorch copy |
| Pi backward self-loop | Editable generic uniform path using GMRES and exact ancestor correction |
| Cross-clade DTS backward | Editable PyTorch copy; production Triton import is available but off by default |
| E adjoint / theta VJP | Editable PyTorch/autograd copy using `torch.func.vjp`, CG, and GMRES |

Note: the production fused uniform backward kernel is imported but disabled by default. The generic notebook path is easier to modify and uses the explicit sparse ancestor correction.

In [ ]:
from __future__ import annotations

import math
from pathlib import Path
from typing import Any, Optional

import torch
from torch import func as tfunc

# Real preprocessing/layout imports. These are intentionally not copied into the notebook.
from gpurec.core.preprocess_cpp import _load_extension
from gpurec.core.model import GeneDataset
from gpurec.core.batching import collate_gene_families, collate_wave, build_wave_layout
from gpurec.core.scheduling import compute_clade_waves
from gpurec.optimization.linear_solvers import _cg, _gmres

# Optional production Triton wrappers. The notebook has PyTorch fallbacks below.
try:
    from gpurec.core.kernels.dts_fused import dts_fused as production_dts_fused
    from gpurec.core.kernels.scatter_lse import seg_logsumexp as production_seg_logsumexp
    from gpurec.core.kernels.wave_step import wave_step_fused as production_wave_step_fused
    from gpurec.core.kernels.wave_backward import (
        wave_backward_uniform_fused as production_wave_backward_uniform_fused,
        dts_cross_backward_fused as production_dts_cross_backward_fused,
    )
    HAS_PRODUCTION_TRITON = True
except Exception as exc:
    production_dts_fused = None
    production_seg_logsumexp = None
    production_wave_step_fused = None
    production_wave_backward_uniform_fused = None
    production_dts_cross_backward_fused = None
    HAS_PRODUCTION_TRITON = False
    TRITON_IMPORT_ERROR = exc

print('HAS_PRODUCTION_TRITON =', HAS_PRODUCTION_TRITON)

## Experiment configuration

Set the paths and mode here. `MODE` controls theta shape; `PIBAR_MODE` should stay `"uniform"` for this notebook.

In [ ]:
SPECIES_TREE = Path('path/to/species.nwk')
GENE_TREES = [Path('path/to/gene1.nwk')]

MODE = 'global'  # 'global', 'specieswise', or 'genewise'
PIBAR_MODE = 'uniform'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE = torch.float32

MAX_ITERS_E = 2000
TOL_E = 1e-8
MAX_ITERS_PI = 2000
TOL_PI = 1e-6

USE_PRODUCTION_TRITON_FORWARD = bool(HAS_PRODUCTION_TRITON and DEVICE.type == 'cuda')
USE_PRODUCTION_TRITON_BACKWARD = False  # Keep off for editable exact-ancestor sandboxing.

print(DEVICE, DTYPE, MODE)

## Preprocessing and layout helpers

These helpers import the real preprocessing extension and real wave-layout builders. The experimental code starts after this section.

In [ ]:
def inspect_preprocessing_extension():
    """Load the C++ preprocessing extension and list public bindings."""
    ext = _load_extension()
    return [name for name in dir(ext) if not name.startswith('_')]


def mode_to_flags(mode: str) -> tuple[bool, bool, bool]:
    if mode == 'global':
        return False, False, False
    if mode == 'specieswise':
        return False, True, False
    if mode == 'genewise':
        return True, False, False
    raise ValueError(f'Unknown mode: {mode!r}')


def make_dataset(
    species_tree: Path | str,
    gene_trees: list[Path | str],
    *,
    mode: str = MODE,
    device: torch.device = DEVICE,
    dtype: torch.dtype = DTYPE,
) -> GeneDataset:
    """Build the real GeneDataset so preprocessing stays identical to production."""
    genewise, specieswise, pairwise = mode_to_flags(mode)
    return GeneDataset(
        species_tree_path=str(species_tree),
        gene_tree_paths=[str(p) for p in gene_trees],
        genewise=genewise,
        specieswise=specieswise,
        pairwise=pairwise,
        dtype=dtype,
        device=device,
    )


def move_tensor_for_static(t: torch.Tensor, *, device: torch.device, dtype: torch.dtype) -> torch.Tensor:
    if t.dtype.is_floating_point:
        return t.to(device=device, dtype=dtype)
    return t.to(device=device)


def build_uniform_static(
    dataset: GeneDataset,
    indices: Optional[list[int]] = None,
    *,
    device: torch.device = DEVICE,
    dtype: torch.dtype = DTYPE,
) -> dict[str, Any]:
    """Build the same static inputs production uses, but return a plain dict for notebook hacking."""
    if indices is None:
        indices = list(range(len(dataset.families)))

    batch_items = []
    family_waves = []
    family_phases = []
    for idx in indices:
        fam = dataset.families[idx]
        batch_items.append({
            'ccp': fam['ccp_helpers'],
            'leaf_row_index': fam['leaf_row_index'],
            'leaf_col_index': fam['leaf_col_index'],
            'root_clade_id': int(fam['root_clade_id']),
        })
        waves_i, phases_i = compute_clade_waves(fam['ccp_helpers'])
        family_waves.append(waves_i)
        family_phases.append(phases_i)

    batched = collate_gene_families(batch_items, dtype=dtype, device=device)
    offsets = [m['clade_offset'] for m in batched['family_meta']]
    cross_waves = collate_wave(family_waves, offsets)

    max_n_waves = max(len(p) for p in family_phases)
    cross_phases = []
    for k in range(max_n_waves):
        phase_k = 1
        for fp in family_phases:
            if k < len(fp):
                phase_k = max(phase_k, fp[k])
        cross_phases.append(phase_k)

    family_clade_counts = [m['C'] for m in batched['family_meta']]
    family_clade_offsets = [m['clade_offset'] for m in batched['family_meta']]

    wave_layout = build_wave_layout(
        waves=cross_waves,
        phases=cross_phases,
        ccp_helpers=batched['ccp'],
        leaf_row_index=batched['leaf_row_index'],
        leaf_col_index=batched['leaf_col_index'],
        root_clade_ids=batched['root_clade_ids'],
        device=device,
        dtype=dtype,
        family_clade_counts=family_clade_counts,
        family_clade_offsets=family_clade_offsets,
    )

    # Keep production behavior for now: consume ancestors_dense and build sparse ancestors_T.
    # This is exactly the seam to replace if preprocessing starts emitting sparse ancestors_T directly.
    species_helpers = {
        k: (move_tensor_for_static(v, device=device, dtype=dtype) if torch.is_tensor(v) and k != 'Recipients_mat' else v)
        for k, v in dataset.species_helpers.items()
    }
    ancestors_T = species_helpers['ancestors_dense'].T.to_sparse_coo()
    unnorm_row_max = dataset.unnorm_row_max.to(device=device, dtype=dtype)

    return {
        'indices': indices,
        'wave_layout': wave_layout,
        'species_helpers': species_helpers,
        'root_clade_ids': batched['root_clade_ids'],
        'unnorm_row_max': unnorm_row_max,
        'ancestors_T': ancestors_T,
        'genewise': bool(dataset.genewise),
        'specieswise': bool(dataset.specieswise),
        'device': device,
        'dtype': dtype,
    }

## Base log-space utilities

These are notebook-local copies of the base-2 log-space operations used by the uniform path.

In [ ]:
NEG_INF = float('-inf')


def safe_log2(x: torch.Tensor) -> torch.Tensor:
    """Return log2(x) where x > 0 and -inf elsewhere."""
    return torch.where(x > 0, torch.log2(torch.clamp_min(x, torch.finfo(x.dtype).tiny)), torch.full_like(x, NEG_INF))


def logsumexp2(x: torch.Tensor, dim: int) -> torch.Tensor:
    m = x.max(dim=dim, keepdim=True).values
    m_safe = torch.where(torch.isfinite(m), m, torch.zeros_like(m))
    s = torch.exp2(x - m_safe).sum(dim=dim, keepdim=True)
    return (torch.log2(s) + m).squeeze(dim)


def logaddexp2(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    m = torch.maximum(a, b)
    m_safe = torch.where(torch.isfinite(m), m, torch.zeros_like(m))
    return torch.log2(torch.exp2(a - m_safe) + torch.exp2(b - m_safe)) + m


def safe_exp2_ratio(num: torch.Tensor, den: torch.Tensor) -> torch.Tensor:
    """Return exp2(num - den), but zero out unreachable/infinite cases."""
    finite = torch.isfinite(num) & torch.isfinite(den)
    return torch.where(finite, torch.exp2(num - den), torch.zeros_like(torch.broadcast_to(num, torch.broadcast_shapes(num.shape, den.shape))))


def log2_softmax_nb(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    return x - logsumexp2(x, dim=dim).unsqueeze(dim)


def gather_E_children_nb(E: torch.Tensor, sp_P_idx: torch.Tensor, child_index: torch.Tensor) -> torch.Tensor:
    """Notebook copy of gather_E_children for E shape [S] or [G, S]."""
    if E.ndim == 1:
        S = E.shape[0]
        out = torch.full((2 * S,), NEG_INF, device=E.device, dtype=E.dtype)
        values = E.index_select(0, child_index)
        out.index_copy_(0, sp_P_idx, values)
        return out
    if E.ndim == 2:
        G, S = E.shape
        out = torch.full((G, 2 * S), NEG_INF, device=E.device, dtype=E.dtype)
        values = E.index_select(1, child_index)
        out.index_copy_(1, sp_P_idx, values)
        return out
    raise ValueError(f'E must be 1D or 2D, got shape {tuple(E.shape)}')

## Parameter extraction: uniform mode

Editable copy of `extract_parameters_uniform`.

In [ ]:
def extract_parameters_uniform_nb(
    theta: torch.Tensor,
    unnorm_row_max: torch.Tensor,
    specieswise: bool,
    genewise: bool = False,
):
    """Extract log-space DTL probabilities and max-transfer vector without materializing [S, S]."""
    if genewise:
        if specieswise:
            G, S, _ = theta.shape
            zeros = theta.new_zeros((G, S, 1))
            result = log2_softmax_nb(torch.cat((zeros, theta), dim=-1), dim=-1)
            log_pS = result[..., 0]
            log_pD = result[..., 1]
            log_pL = result[..., 2]
            log_pT = result[..., 3]
            max_transfer_mat = log_pT + unnorm_row_max
        else:
            zeros = theta.new_zeros((theta.shape[0], 1))
            result = log2_softmax_nb(torch.cat((zeros, theta), dim=-1), dim=-1)
            log_pS = result[:, 0]
            log_pD = result[:, 1]
            log_pL = result[:, 2]
            log_pT = result[:, 3]
            max_transfer_mat = log_pT.unsqueeze(-1) + unnorm_row_max
        return log_pS, log_pD, log_pL, None, max_transfer_mat

    if specieswise:
        S, _ = theta.shape
        zeros = theta.new_zeros((S, 1))
        result = log2_softmax_nb(torch.cat((zeros, theta), dim=-1), dim=-1)
        log_pS = result[..., 0]
        log_pD = result[..., 1]
        log_pL = result[..., 2]
        log_pT = result[..., 3]
        max_transfer_mat = log_pT + unnorm_row_max
    else:
        zeros = theta.new_zeros((1,))
        result = log2_softmax_nb(torch.cat((zeros, theta), dim=-1), dim=-1)
        log_pS = result[0]
        log_pD = result[1]
        log_pL = result[2]
        log_pT = result[3]
        max_transfer_mat = log_pT + unnorm_row_max

    return log_pS, log_pD, log_pL, None, max_transfer_mat

## E fixed point: uniform mode

Editable copies of `E_step` and `E_fixed_point` specialized to uniform `Pibar`/`Ebar`.

In [ ]:
def align_param_to_E_nb(p: torch.Tensor, E_ref: torch.Tensor):
    if not isinstance(p, torch.Tensor):
        return p
    if p.ndim == 0:
        return p
    if E_ref.ndim == 1:
        return p
    G, S = E_ref.shape
    if p.ndim == 1 and p.shape[0] == G:
        return p.unsqueeze(-1)
    return p


def compute_Ebar_uniform_nb(E: torch.Tensor, max_transfer_mat: torch.Tensor, ancestors_T: torch.Tensor) -> torch.Tensor:
    mt = max_transfer_mat.squeeze(-1) if max_transfer_mat.ndim > 1 and max_transfer_mat.shape[-1] == 1 else max_transfer_mat
    max_E = E.max(dim=-1, keepdim=True).values
    expE = torch.exp2(E - max_E)
    expE_2d = expE.unsqueeze(0) if expE.ndim == 1 else expE
    row_sum = expE_2d.sum(dim=-1, keepdim=True)
    ancestor_sum = (expE_2d @ ancestors_T).contiguous()
    linear = row_sum - ancestor_sum
    if expE.ndim == 1:
        return safe_log2(linear.squeeze(0)) + max_E.squeeze(-1) + mt
    return safe_log2(linear) + max_E + mt


def E_step_uniform_nb(
    E: torch.Tensor,
    sp_P_idx: torch.Tensor,
    sp_child12_idx: torch.Tensor,
    log_pS: torch.Tensor,
    log_pD: torch.Tensor,
    log_pL: torch.Tensor,
    max_transfer_mat: torch.Tensor,
    ancestors_T: torch.Tensor,
):
    E_stack = torch.empty((4, *E.shape), dtype=E.dtype, device=E.device)

    E_s12 = gather_E_children_nb(E, sp_P_idx, sp_child12_idx)
    E_s1, E_s2 = torch.chunk(E_s12, 2, dim=-1)
    E_s1 = E_s1.view(E.shape)
    E_s2 = E_s2.view(E.shape)

    pS = align_param_to_E_nb(log_pS, E)
    pD = align_param_to_E_nb(log_pD, E)
    pL = align_param_to_E_nb(log_pL, E)

    Ebar = compute_Ebar_uniform_nb(E, max_transfer_mat, ancestors_T)

    E_stack[0] = pS + E_s1 + E_s2
    E_stack[1] = pD + 2 * E
    E_stack[2] = E + Ebar
    E_stack[3] = pL

    E_new = logsumexp2(E_stack, dim=0)
    return E_new, E_s1, E_s2, Ebar


def E_fixed_point_uniform_nb(
    species_helpers: dict[str, Any],
    log_pS: torch.Tensor,
    log_pD: torch.Tensor,
    log_pL: torch.Tensor,
    max_transfer_mat: torch.Tensor,
    *,
    max_iters: int = MAX_ITERS_E,
    tolerance: float = TOL_E,
    warm_start_E: Optional[torch.Tensor] = None,
    dtype: torch.dtype = DTYPE,
    device: torch.device = DEVICE,
    ancestors_T: torch.Tensor,
):
    S = int(species_helpers['S'])

    G = None
    if isinstance(log_pS, torch.Tensor) and log_pS.ndim == 2:
        G = log_pS.shape[0]
    elif isinstance(log_pS, torch.Tensor) and log_pS.ndim == 1 and log_pS.shape[0] != S:
        G = log_pS.shape[0]
    elif isinstance(log_pD, torch.Tensor) and log_pD.ndim == 1 and log_pD.shape[0] != S:
        G = log_pD.shape[0]
    elif isinstance(log_pL, torch.Tensor) and log_pL.ndim == 1 and log_pL.shape[0] != S:
        G = log_pL.shape[0]

    if warm_start_E is not None:
        E = warm_start_E.detach().clone()
    else:
        E = torch.full((S,) if G is None else (G, S), -1.0, dtype=dtype, device=device)

    E_s1 = torch.full_like(E, NEG_INF)
    E_s2 = torch.full_like(E, NEG_INF)
    Ebar = torch.full_like(E, NEG_INF)
    converged_iter = max_iters

    for iteration in range(max_iters):
        E_new, E_s1, E_s2, Ebar = E_step_uniform_nb(
            E,
            species_helpers['s_P_indexes'],
            species_helpers['s_C12_indexes'],
            log_pS,
            log_pD,
            log_pL,
            max_transfer_mat,
            ancestors_T,
        )
        if torch.abs(E_new - E).max().item() < tolerance:
            converged_iter = iteration + 1
            E = E_new
            break
        E = E_new

    return {'E': E, 'iterations': converged_iter, 'E_s1': E_s1, 'E_s2': E_s2, 'E_bar': Ebar}

## Pi forward helpers

These are editable copies of uniform `Pibar`, cross-clade DTS, and the wave self-loop step. `compute_dts_cross_nb` and `wave_step_nb` can call production Triton wrappers if enabled.

In [ ]:
def make_species_child_vectors_nb(species_helpers: dict[str, Any], *, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    S = int(species_helpers['S'])
    p_cpu = species_helpers['s_P_indexes'].cpu().long()
    c_cpu = species_helpers['s_C12_indexes'].cpu().long()
    mask_c1 = p_cpu < S
    sp_child1_cpu = torch.full((S,), S, dtype=torch.long)
    sp_child2_cpu = torch.full((S,), S, dtype=torch.long)
    sp_child1_cpu[p_cpu[mask_c1]] = c_cpu[mask_c1]
    sp_child2_cpu[p_cpu[~mask_c1] - S] = c_cpu[~mask_c1]
    return sp_child1_cpu.to(device), sp_child2_cpu.to(device)


def compute_Pibar_uniform_nb(Pi_W: torch.Tensor, mt_squeezed: torch.Tensor, ancestors_T: torch.Tensor) -> torch.Tensor:
    Pi_max = Pi_W.max(dim=1, keepdim=True).values
    Pi_exp = torch.exp2(Pi_W - Pi_max)
    row_sum = Pi_exp.sum(dim=1, keepdim=True)
    ancestor_sum = (Pi_exp @ ancestors_T).contiguous()
    return (safe_log2(row_sum - ancestor_sum) + Pi_max + mt_squeezed).contiguous()


def compute_dts_cross_pytorch_nb(
    Pi: torch.Tensor,
    Pibar: torch.Tensor,
    meta: dict[str, Any],
    sp_child1: torch.Tensor,
    sp_child2: torch.Tensor,
    log_pD: torch.Tensor,
    log_pS: torch.Tensor,
    S: int,
    device: torch.device,
    dtype: torch.dtype,
) -> torch.Tensor:
    sl = meta['sl']
    sr = meta['sr']
    wlsp = meta['log_split_probs']
    W = meta['W']
    n_ws = sl.shape[0]

    Pi_l = Pi[sl]
    Pi_r = Pi[sr]
    Pibar_l = Pibar[sl]
    Pibar_r = Pibar[sr]

    Pi_pad = torch.cat([Pi, torch.full((Pi.shape[0], 1), NEG_INF, device=device, dtype=dtype)], dim=1)
    Pi_l_s1 = Pi_pad[sl][:, sp_child1.long()]
    Pi_l_s2 = Pi_pad[sl][:, sp_child2.long()]
    Pi_r_s1 = Pi_pad[sr][:, sp_child1.long()]
    Pi_r_s2 = Pi_pad[sr][:, sp_child2.long()]

    DTS = torch.stack([
        log_pD + Pi_l + Pi_r,
        Pi_l + Pibar_r,
        Pi_r + Pibar_l,
        log_pS + Pi_l_s1 + Pi_r_s2,
        log_pS + Pi_r_s1 + Pi_l_s2,
    ], dim=0)
    dts_term = wlsp + logsumexp2(DTS, dim=0)

    reduce_idx = meta['reduce_idx']
    reduce_expand = reduce_idx.unsqueeze(1).expand(n_ws, S)
    seg_max = torch.full((W, S), NEG_INF, device=device, dtype=dtype)
    seg_max.scatter_reduce_(0, reduce_expand, dts_term.detach(), reduce='amax', include_self=True)
    seg_max_safe = torch.where(seg_max == NEG_INF, torch.zeros_like(seg_max), seg_max)
    shifted = torch.exp2(dts_term - seg_max_safe[reduce_idx])
    seg_sum = torch.zeros((W, S), device=device, dtype=dtype)
    seg_sum.scatter_add_(0, reduce_expand, shifted)
    return safe_log2(seg_sum) + seg_max


def compute_dts_cross_nb(
    Pi: torch.Tensor,
    Pibar: torch.Tensor,
    meta: dict[str, Any],
    sp_child1: torch.Tensor,
    sp_child2: torch.Tensor,
    log_pD: torch.Tensor,
    log_pS: torch.Tensor,
    S: int,
    device: torch.device,
    dtype: torch.dtype,
    *,
    use_triton: bool = USE_PRODUCTION_TRITON_FORWARD,
) -> torch.Tensor:
    if use_triton and HAS_PRODUCTION_TRITON and device.type == 'cuda':
        dts_term = production_dts_fused(
            Pi,
            Pibar,
            meta['sl'],
            meta['sr'],
            sp_child1,
            sp_child2,
            log_pD,
            log_pS,
            meta['log_split_probs'],
        )
        dts_r = torch.full((meta['W'], S), NEG_INF, device=device, dtype=dtype)
        n_eq1 = meta.get('n_eq1', 0)
        n_ge2_clades = meta.get('n_ge2_clades', 0)
        if n_eq1 > 0:
            dts_r[meta['eq1_reduce_idx']] = dts_term[:n_eq1]
        if n_ge2_clades > 0:
            ge2_term = dts_term[n_eq1:].contiguous()
            y_ge2 = production_seg_logsumexp(ge2_term, meta['ge2_ptr'])
            dts_r[meta['ge2_parent_ids']] = y_ge2
        return dts_r

    return compute_dts_cross_pytorch_nb(Pi, Pibar, meta, sp_child1, sp_child2, log_pD, log_pS, S, device, dtype)


def expand_wave_const_nb(t: torch.Tensor, W: int) -> torch.Tensor:
    return t.unsqueeze(0).expand(W, -1) if t.ndim == 1 else t


def wave_step_pytorch_nb(
    Pi_W: torch.Tensor,
    Pibar_W: torch.Tensor,
    DL_const: torch.Tensor,
    Ebar: torch.Tensor,
    E: torch.Tensor,
    SL1_const: torch.Tensor,
    SL2_const: torch.Tensor,
    sp_child1: torch.Tensor,
    sp_child2: torch.Tensor,
    leaf_wt: torch.Tensor,
    dts_r: Optional[torch.Tensor] = None,
) -> torch.Tensor:
    W, S = Pi_W.shape
    DL_w = expand_wave_const_nb(DL_const, W)
    Ebar_w = expand_wave_const_nb(Ebar, W)
    E_w = expand_wave_const_nb(E, W)
    SL1_w = expand_wave_const_nb(SL1_const, W)
    SL2_w = expand_wave_const_nb(SL2_const, W)

    Pi_pad = torch.cat([Pi_W, torch.full((W, 1), NEG_INF, device=Pi_W.device, dtype=Pi_W.dtype)], dim=1)
    Pi_s1 = Pi_pad[:, sp_child1.long()]
    Pi_s2 = Pi_pad[:, sp_child2.long()]

    DTS_L = torch.stack([
        DL_w + Pi_W,
        Pi_W + Ebar_w,
        Pibar_W + E_w,
        SL1_w + Pi_s1,
        SL2_w + Pi_s2,
        leaf_wt,
    ], dim=0)
    DTS_L_term = logsumexp2(DTS_L, dim=0)
    return logaddexp2(dts_r, DTS_L_term) if dts_r is not None else DTS_L_term


def wave_step_nb(
    Pi_W: torch.Tensor,
    Pibar_W: torch.Tensor,
    DL_const: torch.Tensor,
    Ebar: torch.Tensor,
    E: torch.Tensor,
    SL1_const: torch.Tensor,
    SL2_const: torch.Tensor,
    sp_child1: torch.Tensor,
    sp_child2: torch.Tensor,
    leaf_wt: torch.Tensor,
    dts_r: Optional[torch.Tensor] = None,
    *,
    use_triton: bool = USE_PRODUCTION_TRITON_FORWARD,
) -> torch.Tensor:
    if use_triton and HAS_PRODUCTION_TRITON and Pi_W.device.type == 'cuda':
        return production_wave_step_fused(
            Pi_W,
            Pibar_W,
            DL_const,
            Ebar,
            E,
            SL1_const,
            SL2_const,
            sp_child1,
            sp_child2,
            leaf_wt,
            dts_r,
        )
    return wave_step_pytorch_nb(Pi_W, Pibar_W, DL_const, Ebar, E, SL1_const, SL2_const, sp_child1, sp_child2, leaf_wt, dts_r)

## Pi wave forward: uniform mode

Editable copy of the active production structure: compute `Pibar` separately, then apply the wave self-loop step.

In [ ]:
def Pi_wave_forward_uniform_nb(
    wave_layout: dict[str, Any],
    species_helpers: dict[str, Any],
    E: torch.Tensor,
    Ebar: torch.Tensor,
    E_s1: torch.Tensor,
    E_s2: torch.Tensor,
    log_pS: torch.Tensor,
    log_pD: torch.Tensor,
    log_pL: torch.Tensor,
    max_transfer_mat: torch.Tensor,
    *,
    ancestors_T: torch.Tensor,
    device: torch.device = DEVICE,
    dtype: torch.dtype = DTYPE,
    local_iters: int = MAX_ITERS_PI,
    local_tolerance: float = TOL_PI,
    fixed_iters: Optional[int] = None,
    family_idx: Optional[torch.Tensor] = None,
    use_triton: bool = USE_PRODUCTION_TRITON_FORWARD,
):
    ccp_helpers = wave_layout['ccp_helpers']
    leaf_row_index = wave_layout['leaf_row_index']
    leaf_col_index = wave_layout['leaf_col_index']
    wave_metas = wave_layout['wave_metas']

    C = int(ccp_helpers['C'])
    S = int(species_helpers['S'])

    Pi = torch.full((C, S), torch.finfo(dtype).min, dtype=dtype, device=device)
    Pi[leaf_row_index.to(device), leaf_col_index.to(device)] = 0.0
    Pibar = torch.full((C, S), NEG_INF, dtype=dtype, device=device)

    sp_child1, sp_child2 = make_species_child_vectors_nb(species_helpers, device=device)
    batched = family_idx is not None

    pD_for_const = log_pD.unsqueeze(-1) if batched and log_pD.ndim == 1 else log_pD
    pS_for_const = log_pS.unsqueeze(-1) if batched and log_pS.ndim == 1 else log_pS
    DL_const = 1.0 + pD_for_const + E
    SL1_const = pS_for_const + E_s2
    SL2_const = pS_for_const + E_s1

    mt_squeezed = max_transfer_mat.squeeze(-1) if max_transfer_mat.ndim > 1 and max_transfer_mat.shape[-1] == 1 else max_transfer_mat

    leaf_term = None if batched else log_pS + torch.full((C, S), NEG_INF, device=device, dtype=dtype).scatter_(1, leaf_col_index.view(-1, 1), 0.0) if False else None

    def get_leaf_mask(ws, we):
        W = we - ws
        lwt = torch.full((W, S), NEG_INF, device=device, dtype=dtype)
        m = (leaf_row_index >= ws) & (leaf_row_index < we)
        if m.any():
            lwt[leaf_row_index[m] - ws, leaf_col_index[m]] = 0.0
        return lwt

    def get_leaf_wt(ws, we):
        lwt = get_leaf_mask(ws, we)
        if batched:
            fi_w = family_idx[ws:we]
            log_pS_w = log_pS[fi_w]
            if log_pS_w.ndim == 1:
                log_pS_w = log_pS_w.unsqueeze(-1)
            return log_pS_w + lwt
        return log_pS + lwt

    def wave_consts(ws, we):
        if not batched:
            return DL_const, SL1_const, SL2_const, Ebar, E, mt_squeezed
        fi_w = family_idx[ws:we]
        return DL_const[fi_w], SL1_const[fi_w], SL2_const[fi_w], Ebar[fi_w], E[fi_w], mt_squeezed[fi_w]

    def wave_dts_params(meta):
        if not batched:
            return log_pD, log_pS
        ws = meta['start']
        reduce_idx = meta['reduce_idx']
        fi_splits = family_idx[ws + reduce_idx]
        pD = log_pD[fi_splits]
        pS = log_pS[fi_splits]
        if pD.ndim == 1:
            pD = pD.unsqueeze(-1).expand(-1, S).contiguous()
        if pS.ndim == 1:
            pS = pS.unsqueeze(-1).expand(-1, S).contiguous()
        return pD, pS

    use_fixed = fixed_iters is not None
    n_iters = fixed_iters if use_fixed else local_iters
    min_warmup = 0 if use_fixed else 3
    total_iters = 0

    for meta in wave_metas:
        ws = meta['start']
        we = meta['end']
        W = meta['W']

        if meta['has_splits']:
            pD_dts, pS_dts = wave_dts_params(meta)
            dts_r = compute_dts_cross_nb(Pi, Pibar, meta, sp_child1, sp_child2, pD_dts, pS_dts, S, device, dtype, use_triton=use_triton)
        else:
            dts_r = None

        leaf_wt = get_leaf_wt(ws, we)
        DL_w, SL1_w, SL2_w, Ebar_w, E_w, mt_w = wave_consts(ws, we)

        for local_iter in range(n_iters):
            total_iters += 1
            Pi_W = Pi[ws:we]
            Pibar_W = compute_Pibar_uniform_nb(Pi_W, mt_w, ancestors_T)
            Pi_new = wave_step_nb(Pi_W, Pibar_W, DL_w, Ebar_w, E_w, SL1_w, SL2_w, sp_child1, sp_child2, leaf_wt, dts_r, use_triton=use_triton)

            if not use_fixed and local_iter >= min_warmup:
                significant = Pi_new > -100.0
                if not significant.any() or torch.abs(Pi_new - Pi_W)[significant].max().item() < local_tolerance:
                    Pi[ws:we] = Pi_new
                    Pibar[ws:we] = Pibar_W
                    break

            Pi[ws:we] = Pi_new
            Pibar[ws:we] = Pibar_W

    perm = wave_layout['perm']
    return {
        'Pi': Pi[perm],
        'iterations': total_iters,
        'Pi_wave_ordered': Pi,
        'Pibar_wave_ordered': Pibar,
    }

## Likelihood

Editable copy of `compute_log_likelihood`. It returns NLL, not positive log likelihood.

In [ ]:
def compute_log_likelihood_nb(Pi: torch.Tensor, E: torch.Tensor, root_clade_idx: torch.Tensor) -> torch.Tensor:
    root_probs = Pi[root_clade_idx, :]
    numerator = logsumexp2(root_probs, dim=-1) - math.log2(Pi.shape[-1])
    denominator = torch.log2(1.0 - torch.exp2(E).mean(dim=-1))
    return -(numerator - denominator)


def forward_uniform_sandbox(
    static: dict[str, Any],
    theta: torch.Tensor,
    *,
    warm_E: Optional[torch.Tensor] = None,
    use_triton: bool = USE_PRODUCTION_TRITON_FORWARD,
):
    log_pS, log_pD, log_pL, _, mt = extract_parameters_uniform_nb(
        theta,
        static['unnorm_row_max'],
        specieswise=static['specieswise'],
        genewise=static['genewise'],
    )
    E_out = E_fixed_point_uniform_nb(
        static['species_helpers'],
        log_pS,
        log_pD,
        log_pL,
        mt,
        max_iters=MAX_ITERS_E,
        tolerance=TOL_E,
        warm_start_E=warm_E,
        dtype=static['dtype'],
        device=static['device'],
        ancestors_T=static['ancestors_T'],
    )
    Pi_out = Pi_wave_forward_uniform_nb(
        static['wave_layout'],
        static['species_helpers'],
        E_out['E'],
        E_out['E_bar'],
        E_out['E_s1'],
        E_out['E_s2'],
        log_pS,
        log_pD,
        log_pL,
        mt,
        ancestors_T=static['ancestors_T'],
        device=static['device'],
        dtype=static['dtype'],
        family_idx=static['wave_layout'].get('family_idx') if static['genewise'] else None,
        use_triton=use_triton,
    )
    nll_vec = compute_log_likelihood_nb(Pi_out['Pi'], E_out['E'], static['root_clade_ids'])
    return {
        'nll_vec': nll_vec,
        'nll': nll_vec.sum(),
        'log_pS': log_pS,
        'log_pD': log_pD,
        'log_pL': log_pL,
        'mt': mt,
        'E_out': E_out,
        'Pi_out': Pi_out,
    }

## Pi backward helpers

Editable copy of the generic uniform backward pieces: self-loop VJP precompute, exact sparse ancestor correction, and GMRES solve.

In [ ]:
def self_loop_vjp_precompute_uniform_nb(
    Pi_star: torch.Tensor,
    Pibar_star: torch.Tensor,
    dts_r: Optional[torch.Tensor],
    mt_w: torch.Tensor,
    DL_w: torch.Tensor,
    Ebar_w: torch.Tensor,
    E_w: torch.Tensor,
    SL1_w: torch.Tensor,
    SL2_w: torch.Tensor,
    sp_child1: torch.Tensor,
    sp_child2: torch.Tensor,
    leaf_wt: torch.Tensor,
    S: int,
    ancestors_T: torch.Tensor,
):
    W = Pi_star.shape[0]
    device, dtype = Pi_star.device, Pi_star.dtype

    mt = expand_wave_const_nb(mt_w, W)
    DL = expand_wave_const_nb(DL_w, W)
    Ebar = expand_wave_const_nb(Ebar_w, W)
    E = expand_wave_const_nb(E_w, W)
    SL1 = expand_wave_const_nb(SL1_w, W)
    SL2 = expand_wave_const_nb(SL2_w, W)

    Pi_max = Pi_star.max(dim=1, keepdim=True).values
    p_prime = torch.exp2(Pi_star - Pi_max)
    row_sum = p_prime.sum(dim=1, keepdim=True)
    anc_sum = p_prime @ ancestors_T
    pibar_denom = row_sum - anc_sum

    Pi_pad = torch.cat([Pi_star, torch.full((W, 1), NEG_INF, device=device, dtype=dtype)], dim=1)
    Pi_s1 = Pi_pad[:, sp_child1.long()]
    Pi_s2 = Pi_pad[:, sp_child2.long()]

    terms = torch.stack([
        DL + Pi_star,
        Pi_star + Ebar,
        Pibar_star + E,
        SL1 + Pi_s1,
        SL2 + Pi_s2,
        leaf_wt,
    ], dim=0)
    DTS_L = logsumexp2(terms, dim=0)

    if dts_r is not None:
        Pi_new = logaddexp2(dts_r, DTS_L)
        w_L = safe_exp2_ratio(DTS_L, Pi_new)
    else:
        w_L = torch.ones(W, S, device=device, dtype=dtype)

    w_terms = safe_exp2_ratio(terms, DTS_L.unsqueeze(0))
    pos = pibar_denom > 0
    inv_denom = torch.where(pos, 1.0 / torch.where(pos, pibar_denom, torch.ones_like(pibar_denom)), torch.zeros_like(pibar_denom))

    sc1 = sp_child1.long()
    sc2 = sp_child2.long()
    valid1 = sc1 < S
    valid2 = sc2 < S

    result = {
        'w_L': w_L,
        'w_terms': w_terms,
        'p_prime': p_prime,
        'pibar_inv_denom': inv_denom,
    }
    if valid1.any():
        result['sc1_valid'] = valid1
        result['sc1_idx'] = sc1[valid1].unsqueeze(0)
    if valid2.any():
        result['sc2_valid'] = valid2
        result['sc2_idx'] = sc2[valid2].unsqueeze(0)
    return result


def self_loop_Jt_apply_uniform_nb(v: torch.Tensor, ingredients: dict[str, torch.Tensor], sp_child1: torch.Tensor, sp_child2: torch.Tensor, S: int, W: int, ancestors_T: torch.Tensor) -> torch.Tensor:
    w_L = ingredients['w_L']
    w_terms = ingredients['w_terms']
    p_prime = ingredients['p_prime']

    alpha = v * w_L
    result = alpha * (w_terms[0] + w_terms[1])

    v_Pibar = alpha * w_terms[2]
    u_d = v_Pibar * ingredients['pibar_inv_denom']
    A = u_d.sum(dim=1, keepdim=True)
    correction = (ancestors_T @ u_d.T).T
    result = result + p_prime * (A - correction)

    sc1_valid = ingredients.get('sc1_valid')
    sc2_valid = ingredients.get('sc2_valid')
    sc1_idx = ingredients.get('sc1_idx')
    sc2_idx = ingredients.get('sc2_idx')

    if sc1_valid is not None:
        src = alpha * w_terms[3]
        idx = sc1_idx.expand(W, -1) if sc1_idx.shape[0] == 1 else sc1_idx
        result.scatter_add_(1, idx, src[:, sc1_valid])
    if sc2_valid is not None:
        src = alpha * w_terms[4]
        idx = sc2_idx.expand(W, -1) if sc2_idx.shape[0] == 1 else sc2_idx
        result.scatter_add_(1, idx, src[:, sc2_valid])

    return result


def gmres_self_loop_solve_uniform_nb(
    rhs: torch.Tensor,
    ingredients: dict[str, torch.Tensor],
    sp_child1: torch.Tensor,
    sp_child2: torch.Tensor,
    S: int,
    W: int,
    ancestors_T: torch.Tensor,
    *,
    max_iters: int = 5,
    tol: float = 1e-8,
) -> torch.Tensor:
    n = W * S
    b = rhs.reshape(n)
    beta = b.norm()
    if beta < 1e-30:
        return rhs.clone()

    V = [b / beta]
    H = torch.zeros(max_iters + 1, max_iters, device=rhs.device, dtype=rhs.dtype)
    cs = torch.zeros(max_iters, device=rhs.device, dtype=rhs.dtype)
    sn = torch.zeros(max_iters, device=rhs.device, dtype=rhs.dtype)
    g = torch.zeros(max_iters + 1, device=rhs.device, dtype=rhs.dtype)
    g[0] = beta

    converged_j = 0
    for j in range(max_iters):
        vj_2d = V[j].reshape(W, S)
        Jt_vj = self_loop_Jt_apply_uniform_nb(vj_2d, ingredients, sp_child1, sp_child2, S, W, ancestors_T)
        w = (vj_2d - Jt_vj).reshape(n)

        for i in range(j + 1):
            H[i, j] = w.dot(V[i])
            w = w - H[i, j] * V[i]
        H[j + 1, j] = w.norm()

        if H[j + 1, j] > 1e-14:
            V.append(w / H[j + 1, j])
        else:
            V.append(torch.zeros_like(w))

        for i in range(j):
            temp = cs[i] * H[i, j] + sn[i] * H[i + 1, j]
            H[i + 1, j] = -sn[i] * H[i, j] + cs[i] * H[i + 1, j]
            H[i, j] = temp

        denom = (H[j, j] ** 2 + H[j + 1, j] ** 2).sqrt()
        if denom > 1e-14:
            cs[j] = H[j, j] / denom
            sn[j] = H[j + 1, j] / denom
        else:
            cs[j] = 1.0
            sn[j] = 0.0

        H[j, j] = cs[j] * H[j, j] + sn[j] * H[j + 1, j]
        H[j + 1, j] = 0.0
        temp = cs[j] * g[j] + sn[j] * g[j + 1]
        g[j + 1] = -sn[j] * g[j] + cs[j] * g[j + 1]
        g[j] = temp

        converged_j = j + 1
        if abs(float(g[j + 1])) / float(beta) < tol:
            break

    m = converged_j
    y = torch.zeros(m, device=rhs.device, dtype=rhs.dtype)
    for i in range(m - 1, -1, -1):
        y[i] = (g[i] - H[i, i + 1:m] @ y[i + 1:m]) / H[i, i] if H[i, i].abs() > 1e-14 else 0.0

    v = torch.zeros(n, device=rhs.device, dtype=rhs.dtype)
    for i in range(m):
        v = v + float(y[i]) * V[i]
    return v.reshape(W, S)

## Pi backward: uniform mode

Editable generic version of `Pi_wave_backward` specialized to uniform mode. It returns the same gradient dictionary shape used by the E adjoint code.

In [ ]:
@torch.no_grad()
def Pi_wave_backward_uniform_nb(
    wave_layout: dict[str, Any],
    Pi_star_wave: torch.Tensor,
    Pibar_star_wave: torch.Tensor,
    E: torch.Tensor,
    Ebar: torch.Tensor,
    E_s1: torch.Tensor,
    E_s2: torch.Tensor,
    log_pS: torch.Tensor,
    log_pD: torch.Tensor,
    log_pL: torch.Tensor,
    max_transfer_mat: torch.Tensor,
    species_helpers: dict[str, Any],
    root_clade_ids_perm: torch.Tensor,
    *,
    ancestors_T: torch.Tensor,
    device: torch.device = DEVICE,
    dtype: torch.dtype = DTYPE,
    pruning_threshold: float = 1e-6,
    use_pruning: bool = True,
    family_idx: Optional[torch.Tensor] = None,
):
    wave_metas = wave_layout['wave_metas']
    C, S = Pi_star_wave.shape
    K = len(wave_metas)
    sp_child1, sp_child2 = make_species_child_vectors_nb(species_helpers, device=device)

    auto_wrapped = family_idx is None
    if auto_wrapped:
        family_idx = torch.zeros(C, dtype=torch.long, device=device)
        E = E.unsqueeze(0)
        Ebar = Ebar.unsqueeze(0)
        E_s1 = E_s1.unsqueeze(0)
        E_s2 = E_s2.unsqueeze(0)
        log_pS = log_pS.unsqueeze(0)
        log_pD = log_pD.unsqueeze(0)
        log_pL = log_pL.unsqueeze(0)
        max_transfer_mat = max_transfer_mat.unsqueeze(0)

    mt_squeezed = max_transfer_mat.squeeze(-1) if max_transfer_mat.ndim > 2 and max_transfer_mat.shape[-1] == 1 else max_transfer_mat

    def to_clade(p):
        r = p[family_idx]
        return r.unsqueeze(-1) if r.ndim == 1 else r

    mt_clade = to_clade(mt_squeezed)
    E_clade = to_clade(E)
    Ebar_clade = to_clade(Ebar)
    log_pD_clade = to_clade(log_pD)
    log_pS_clade = to_clade(log_pS)
    DL_const = 1.0 + log_pD_clade + E_clade
    SL1_const = log_pS_clade + to_clade(E_s2)
    SL2_const = log_pS_clade + to_clade(E_s1)

    leaf_row_index = wave_layout['leaf_row_index']
    leaf_col_index = wave_layout['leaf_col_index']

    def get_leaf_mask(ws, we):
        W = we - ws
        lwt = torch.full((W, S), NEG_INF, device=device, dtype=dtype)
        mask = (leaf_row_index >= ws) & (leaf_row_index < we)
        if mask.any():
            lwt[leaf_row_index[mask] - ws, leaf_col_index[mask]] = 0.0
        return lwt

    def get_leaf_wt(ws, we):
        return log_pS_clade[ws:we] + get_leaf_mask(ws, we)

    accumulated_rhs = torch.zeros(C, S, device=device, dtype=dtype)
    for r in root_clade_ids_perm:
        r = int(r)
        root_Pi = Pi_star_wave[r]
        lse = logsumexp2(root_Pi, dim=0)
        accumulated_rhs[r] = -safe_exp2_ratio(root_Pi, lse)

    grad_log_pD = torch.zeros_like(log_pD)
    grad_log_pS = torch.zeros_like(log_pS)
    grad_mt = torch.zeros_like(mt_squeezed)
    grad_E_acc = torch.zeros_like(E)
    grad_Ebar_acc = torch.zeros_like(Ebar)
    grad_E_s1_acc = torch.zeros_like(E_s1)
    grad_E_s2_acc = torch.zeros_like(E_s2)

    n_waves_skipped = 0
    n_clades_skipped = 0

    for k in range(K - 1, -1, -1):
        meta = wave_metas[k]
        ws = meta['start']
        we = meta['end']
        W = meta['W']

        rhs_k = accumulated_rhs[ws:we].clone()
        clade_max = rhs_k.abs().max(dim=1).values
        active_mask = clade_max >= pruning_threshold if use_pruning else clade_max > 0
        if not active_mask.any():
            n_waves_skipped += 1
            n_clades_skipped += W
            continue
        n_active = int(active_mask.sum().item())
        n_clades_skipped += W - n_active

        Pi_W_star = Pi_star_wave[ws:we].detach()
        Pibar_W_star = Pibar_star_wave[ws:we].detach()
        leaf_wt = get_leaf_wt(ws, we)

        if meta['has_splits']:
            reduce_idx = meta['reduce_idx']
            log_pD_dts = log_pD_clade[ws + reduce_idx]
            log_pS_dts = log_pS_clade[ws + reduce_idx]
            dts_r = compute_dts_cross_pytorch_nb(
                Pi_star_wave.detach(),
                Pibar_star_wave.detach(),
                meta,
                sp_child1,
                sp_child2,
                log_pD_dts,
                log_pS_dts,
                S,
                device,
                dtype,
            )
        else:
            dts_r = None

        mt_w = mt_clade[ws:we]
        DL_w = DL_const[ws:we]
        E_w = E_clade[ws:we]
        Ebar_w = Ebar_clade[ws:we]
        SL1_w = SL1_const[ws:we]
        SL2_w = SL2_const[ws:we]

        ingredients = self_loop_vjp_precompute_uniform_nb(
            Pi_W_star,
            Pibar_W_star,
            dts_r,
            mt_w,
            DL_w,
            Ebar_w,
            E_w,
            SL1_w,
            SL2_w,
            sp_child1,
            sp_child2,
            leaf_wt,
            S,
            ancestors_T,
        )

        use_compact = n_active < W
        if use_compact:
            active_idx = active_mask.nonzero(as_tuple=True)[0]
            rhs_active = rhs_k[active_idx]
            solve_ing = {
                'w_L': ingredients['w_L'][active_idx],
                'w_terms': ingredients['w_terms'][:, active_idx],
                'p_prime': ingredients['p_prime'][active_idx],
                'pibar_inv_denom': ingredients['pibar_inv_denom'][active_idx],
            }
            for key in ('sc1_valid', 'sc1_idx', 'sc2_valid', 'sc2_idx'):
                if key in ingredients:
                    solve_ing[key] = ingredients[key]
            solve_W = n_active
        else:
            active_idx = None
            rhs_active = rhs_k
            solve_ing = ingredients
            solve_W = W

        v_k = gmres_self_loop_solve_uniform_nb(rhs_active, solve_ing, sp_child1, sp_child2, S, solve_W, ancestors_T)
        if use_compact:
            v_k_full = torch.zeros(W, S, device=device, dtype=dtype)
            v_k_full[active_idx] = v_k
            v_k = v_k_full

        fi_w = family_idx[ws:we]
        fi_expand = fi_w.unsqueeze(1).expand(W, S)

        def scatter_accum(acc, contrib):
            if acc.ndim == 1:
                acc.scatter_add_(0, fi_w, contrib.sum(dim=1))
            else:
                acc.scatter_add_(0, fi_expand, contrib)

        alpha_full = v_k * ingredients['w_L']
        wt = ingredients['w_terms']
        aw0 = alpha_full * wt[0]
        aw1 = alpha_full * wt[1]
        aw2 = alpha_full * wt[2]
        aw3 = alpha_full * wt[3]
        aw4 = alpha_full * wt[4]
        aw5 = alpha_full * wt[5]

        scatter_accum(grad_log_pD, aw0)
        scatter_accum(grad_log_pS, aw3 + aw4 + aw5)
        scatter_accum(grad_E_acc, aw0 + aw2)
        scatter_accum(grad_Ebar_acc, aw1)
        scatter_accum(grad_E_s1_acc, aw4)
        scatter_accum(grad_E_s2_acc, aw3)
        scatter_accum(grad_mt, aw2)

        if meta['has_splits'] and dts_r is not None:
            sl = meta['sl']
            sr = meta['sr']
            wlsp = meta['log_split_probs']
            reduce_idx = meta['reduce_idx']
            n_ws = sl.shape[0]

            Pi_l = Pi_star_wave[sl]
            Pi_r = Pi_star_wave[sr]
            Pibar_l = Pibar_star_wave[sl]
            Pibar_r = Pibar_star_wave[sr]
            neg_inf_col = torch.full((Pi_star_wave.shape[0], 1), NEG_INF, device=device, dtype=dtype)
            Pi_col_pad = torch.cat([Pi_star_wave, neg_inf_col], dim=1)
            Pi_l_s1 = Pi_col_pad[sl][:, sp_child1.long()]
            Pi_l_s2 = Pi_col_pad[sl][:, sp_child2.long()]
            Pi_r_s1 = Pi_col_pad[sr][:, sp_child1.long()]
            Pi_r_s2 = Pi_col_pad[sr][:, sp_child2.long()]

            fi_splits = family_idx[ws + reduce_idx]
            pD_s = log_pD[fi_splits]
            if pD_s.ndim == 1:
                pD_s = pD_s.unsqueeze(-1)
            pS_s = log_pS[fi_splits]
            if pS_s.ndim == 1:
                pS_s = pS_s.unsqueeze(-1)

            DTS_5 = torch.stack([
                pD_s + Pi_l + Pi_r,
                Pi_l + Pibar_r,
                Pi_r + Pibar_l,
                pS_s + Pi_l_s1 + Pi_r_s2,
                pS_s + Pi_r_s1 + Pi_l_s2,
            ], dim=0)

            Pi_parent = Pi_W_star[reduce_idx]
            combined = wlsp + DTS_5
            v_k_parent = v_k[reduce_idx]
            grad_DTS_5 = v_k_parent.unsqueeze(0) * safe_exp2_ratio(combined, Pi_parent.unsqueeze(0))

            fi_split_expand = fi_splits.unsqueeze(1).expand(n_ws, S)
            if grad_log_pD.ndim == 1:
                grad_log_pD.scatter_add_(0, fi_splits, grad_DTS_5[0].sum(dim=1))
                grad_log_pS.scatter_add_(0, fi_splits, (grad_DTS_5[3] + grad_DTS_5[4]).sum(dim=1))
            else:
                grad_log_pD.scatter_add_(0, fi_split_expand, grad_DTS_5[0])
                grad_log_pS.scatter_add_(0, fi_split_expand, grad_DTS_5[3] + grad_DTS_5[4])

            child_ids_dts = torch.cat([sl, sr])
            fi_ch = family_idx[child_ids_dts]
            fi_ch_expand = fi_ch.unsqueeze(1).expand(2 * n_ws, S)
            grad_mt.scatter_add_(0, fi_ch_expand, torch.cat([grad_DTS_5[2], grad_DTS_5[1]], dim=0))

            grad_Pi_l = grad_DTS_5[0] + grad_DTS_5[1]
            grad_Pi_r = grad_DTS_5[0] + grad_DTS_5[2]
            grad_Pibar_l = grad_DTS_5[2]
            grad_Pibar_r = grad_DTS_5[1]

            sc1 = sp_child1.long()
            sc2 = sp_child2.long()
            valid1 = sc1 < S
            valid2 = sc2 < S
            if valid1.any():
                idx1 = sc1[valid1]
                grad_Pi_l.scatter_add_(1, idx1.unsqueeze(0).expand(n_ws, -1), grad_DTS_5[3][:, valid1])
                grad_Pi_r.scatter_add_(1, idx1.unsqueeze(0).expand(n_ws, -1), grad_DTS_5[4][:, valid1])
            if valid2.any():
                idx2 = sc2[valid2]
                grad_Pi_r.scatter_add_(1, idx2.unsqueeze(0).expand(n_ws, -1), grad_DTS_5[3][:, valid2])
                grad_Pi_l.scatter_add_(1, idx2.unsqueeze(0).expand(n_ws, -1), grad_DTS_5[4][:, valid2])

            accumulated_rhs.index_add_(0, sl, grad_Pi_l)
            accumulated_rhs.index_add_(0, sr, grad_Pi_r)

            all_children = torch.cat([sl, sr])
            all_pibar_grad = torch.cat([grad_Pibar_l, grad_Pibar_r])
            nz = all_pibar_grad.abs().sum(dim=1) > 0
            if nz.any():
                nz_children = all_children[nz]
                u = all_pibar_grad[nz]
                Pi_ch = Pi_star_wave[nz_children]
                Pi_max = Pi_ch.max(dim=1, keepdim=True).values
                p_prime = torch.exp2(Pi_ch - Pi_max)
                anc_sum = p_prime @ ancestors_T
                denom = p_prime.sum(dim=1, keepdim=True) - anc_sum
                denom_safe = torch.where(denom > 0, denom, torch.ones_like(denom))
                u_d = torch.where(denom > 0, u / denom_safe, torch.zeros_like(u))
                A = u_d.sum(dim=1, keepdim=True)
                correction = (ancestors_T @ u_d.T).T
                pi_from_pibar = p_prime * (A - correction)
                accumulated_rhs.index_add_(0, nz_children, pi_from_pibar)

    result = {
        'v_Pi': accumulated_rhs,
        'grad_E': grad_E_acc,
        'grad_Ebar': grad_Ebar_acc,
        'grad_E_s1': grad_E_s1_acc,
        'grad_E_s2': grad_E_s2_acc,
        'grad_log_pD': grad_log_pD,
        'grad_log_pS': grad_log_pS,
        'grad_max_transfer_mat': grad_mt,
        'n_waves_total': K,
        'n_waves_skipped': n_waves_skipped,
        'n_waves_processed': K - n_waves_skipped,
        'n_clades_total': C,
        'n_clades_skipped': n_clades_skipped,
        'n_clades_active': C - n_clades_skipped,
    }

    if auto_wrapped:
        for key in ('grad_E', 'grad_Ebar', 'grad_E_s1', 'grad_E_s2', 'grad_log_pD', 'grad_log_pS', 'grad_max_transfer_mat'):
            result[key] = result[key][0]
    return result

## E adjoint and theta VJP

Editable copy of the uniform branch of `_e_adjoint_and_theta_vjp`.

In [ ]:
def e_adjoint_and_theta_vjp_uniform_nb(
    pi_bwd: dict[str, torch.Tensor],
    E_star: torch.Tensor,
    Ebar: torch.Tensor,
    E_s1: torch.Tensor,
    E_s2: torch.Tensor,
    log_pS: torch.Tensor,
    log_pD: torch.Tensor,
    log_pL: torch.Tensor,
    max_transfer_mat: torch.Tensor,
    species_helpers: dict[str, Any],
    root_clade_ids_perm: torch.Tensor,
    theta: torch.Tensor,
    unnorm_row_max: torch.Tensor,
    specieswise: bool,
    *,
    genewise: bool = False,
    ancestors_T: torch.Tensor,
    cg_tol: float = 1e-8,
    cg_maxiter: int = 500,
    gmres_restart: int = 40,
):
    sp_P_idx = species_helpers['s_P_indexes']
    sp_c12_idx = species_helpers['s_C12_indexes']

    n_fam = root_clade_ids_perm.numel()
    E_req_d = E_star.detach().requires_grad_(True)
    with torch.enable_grad():
        mean_E_exp = torch.exp2(E_req_d).mean(dim=-1)
        denom = torch.log2(1.0 - mean_E_exp)
        if E_req_d.ndim > 1 and E_req_d.shape[0] == n_fam:
            direct_obj = denom.sum()
        else:
            direct_obj = (n_fam * denom).sum() if denom.ndim > 0 else (n_fam * denom)
        direct_dNLL_dE = torch.autograd.grad(direct_obj, E_req_d)[0]

    q_E = pi_bwd['grad_E'].clone() + direct_dNLL_dE

    if pi_bwd['grad_Ebar'].abs().max() > 0:
        E_req2 = E_star.detach().requires_grad_(True)
        with torch.enable_grad():
            Ebar_recomp = compute_Ebar_uniform_nb(E_req2, max_transfer_mat, ancestors_T)
            ebar_to_e = torch.autograd.grad(Ebar_recomp, E_req2, grad_outputs=pi_bwd['grad_Ebar'], retain_graph=False)[0]
        q_E = q_E + ebar_to_e

    if pi_bwd['grad_E_s1'].abs().max() > 0 or pi_bwd['grad_E_s2'].abs().max() > 0:
        E_req3 = E_star.detach().requires_grad_(True)
        with torch.enable_grad():
            E_s12 = gather_E_children_nb(E_req3, sp_P_idx, sp_c12_idx)
            E_s1_r, E_s2_r = torch.chunk(E_s12, 2, dim=-1)
            E_s1_r = E_s1_r.view(E_req3.shape)
            E_s2_r = E_s2_r.view(E_req3.shape)
            total = (E_s1_r * pi_bwd['grad_E_s1']).sum() + (E_s2_r * pi_bwd['grad_E_s2']).sum()
            es_to_e = torch.autograd.grad(total, E_req3, retain_graph=False)[0]
        q_E = q_E + es_to_e

    def G_E_fun(E_in):
        return E_step_uniform_nb(E_in, sp_P_idx, sp_c12_idx, log_pS, log_pD, log_pL, max_transfer_mat, ancestors_T)[0]

    E_req_g = E_star.detach().requires_grad_(True)
    with torch.enable_grad():
        _, vjpG = tfunc.vjp(G_E_fun, E_req_g)

    E_shape = E_star.shape
    q_flat = q_E.reshape(-1)

    def AG_flat(w_flat):
        wE = w_flat.view(E_shape).contiguous()
        (gE,) = vjpG(wE.clone())
        return (wE - gE).reshape(-1)

    w_flat, statsG, okG = _cg(AG_flat, q_flat, tol=cg_tol, maxiter=cg_maxiter)
    if not okG:
        w_flat, statsG = _gmres(AG_flat, q_flat, tol=cg_tol, restart=gmres_restart, maxiter=cg_maxiter)
        statsG.fallback_used = True
    wE = w_flat.view(E_shape)

    grad_mt_total = pi_bwd['grad_max_transfer_mat'] + pi_bwd['grad_Ebar']

    theta_req = theta.detach().requires_grad_(True)
    with torch.enable_grad():
        log_pS_r, log_pD_r, log_pL_r, _, mt_r = extract_parameters_uniform_nb(
            theta_req,
            unnorm_row_max,
            specieswise=specieswise,
            genewise=genewise,
        )
        param_loss = (
            (log_pS_r * pi_bwd['grad_log_pS']).sum()
            + (log_pD_r * pi_bwd['grad_log_pD']).sum()
            + (mt_r * grad_mt_total).sum()
        )
        grad_theta_pi = torch.autograd.grad(param_loss, theta_req, retain_graph=False)[0]

    theta_req2 = theta.detach().requires_grad_(True)
    with torch.enable_grad():
        log_pS_r2, log_pD_r2, log_pL_r2, _, mt_r2 = extract_parameters_uniform_nb(
            theta_req2,
            unnorm_row_max,
            specieswise=specieswise,
            genewise=genewise,
        )
        E_from_theta = E_step_uniform_nb(E_star.detach(), sp_P_idx, sp_c12_idx, log_pS_r2, log_pD_r2, log_pL_r2, mt_r2, ancestors_T)[0]
        gtheta_E = torch.autograd.grad(E_from_theta, theta_req2, grad_outputs=wE, retain_graph=False)[0]

    return (grad_theta_pi + gtheta_E).detach(), statsG

## End-to-end backward wrapper

This uses the saved output from `forward_uniform_sandbox` and returns the theta gradient.

In [ ]:
def backward_uniform_sandbox(
    static: dict[str, Any],
    theta: torch.Tensor,
    fwd: dict[str, Any],
    *,
    pruning_threshold: float = 1e-6,
    use_pruning: bool = True,
):
    Pi_out = fwd['Pi_out']
    E_out = fwd['E_out']

    pi_bwd = Pi_wave_backward_uniform_nb(
        static['wave_layout'],
        Pi_out['Pi_wave_ordered'],
        Pi_out['Pibar_wave_ordered'],
        E_out['E'],
        E_out['E_bar'],
        E_out['E_s1'],
        E_out['E_s2'],
        fwd['log_pS'],
        fwd['log_pD'],
        fwd['log_pL'],
        fwd['mt'],
        static['species_helpers'],
        static['wave_layout']['root_clade_ids'],
        ancestors_T=static['ancestors_T'],
        device=static['device'],
        dtype=static['dtype'],
        pruning_threshold=pruning_threshold,
        use_pruning=use_pruning,
        family_idx=static['wave_layout'].get('family_idx') if static['genewise'] else None,
    )

    grad_theta, stats = e_adjoint_and_theta_vjp_uniform_nb(
        pi_bwd,
        E_out['E'],
        E_out['E_bar'],
        E_out['E_s1'],
        E_out['E_s2'],
        fwd['log_pS'],
        fwd['log_pD'],
        fwd['log_pL'],
        fwd['mt'],
        static['species_helpers'],
        static['wave_layout']['root_clade_ids'],
        theta,
        static['unnorm_row_max'],
        static['specieswise'],
        genewise=static['genewise'],
        ancestors_T=static['ancestors_T'],
    )
    return {'grad_theta': grad_theta, 'pi_bwd': pi_bwd, 'stats': stats}

## Example usage

Uncomment and edit this cell after setting `SPECIES_TREE` and `GENE_TREES`. Nothing in this notebook writes to production code.

In [ ]:
# dataset = make_dataset(SPECIES_TREE, GENE_TREES, mode=MODE, device=DEVICE, dtype=DTYPE)
# static = build_uniform_static(dataset, device=DEVICE, dtype=DTYPE)
#
# if MODE == 'global':
#     theta = torch.full((3,), math.log2(1e-10), dtype=DTYPE, device=DEVICE)
# elif MODE == 'specieswise':
#     S = int(static['species_helpers']['S'])
#     theta = torch.full((S, 3), math.log2(1e-10), dtype=DTYPE, device=DEVICE)
# elif MODE == 'genewise':
#     G = len(GENE_TREES)
#     theta = torch.full((G, 3), math.log2(1e-10), dtype=DTYPE, device=DEVICE)
#
# fwd = forward_uniform_sandbox(static, theta, use_triton=USE_PRODUCTION_TRITON_FORWARD)
# print('NLL:', float(fwd['nll']))
#
# bwd = backward_uniform_sandbox(static, theta, fwd)
# print('grad_theta shape:', tuple(bwd['grad_theta'].shape))
# print('grad_theta:', bwd['grad_theta'])